In [ ]:
from datasets import load_dataset
import re

print("Loading News Dataset...")
dataset = load_dataset("ag_news", split="train")

politics_data = dataset.filter(lambda x: x['label'] == 0)

Loading News Dataset...


In [ ]:
cda_dataset=[]
for row in dataset:
  text = row["text"]
  text = text.replace("Black", "TEMP_TOKEN_CAP")
  text = text.replace("black", "temp_token_lower")
  
  text = text.replace("White", "Black")
  text = text.replace("white", "black")
  
  text = text.replace("TEMP_TOKEN_CAP", "White")
  text = text.replace("temp_token_lower", "white")
  
  cda_dataset.append(text)


In [ ]:
import torch
from transformers import BertTokenizer, BertForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import Dataset
model_name = "bert-base-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name)

hf_dataset = Dataset.from_dict({"text": cda_dataset})
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

In [5]:
drive_path="./racial_debiased_bert"

In [ ]:
training_args = TrainingArguments(
    output_dir=drive_path,
    overwrite_output_dir=True,
    num_train_epochs=3,       
    per_device_train_batch_size=8,
    save_steps=5000,
    learning_rate=2e-5,        
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

print("Starting Fine-Tuning...")
trainer.train()


trainer.save_model(drive_path)
tokenizer.save_pretrained(drive_path)
print(f"Model saved to {drive_path}")

Starting Fine-Tuning...


/mnt/home/madired3/.local/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
500,2.304300
1000,2.061500
1500,1.973800
2000,1.919100
2500,1.870400
3000,1.845400
3500,1.823600
4000,1.788200
4500,1.761300
5000,1.758400


/mnt/home/madired3/.local/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/mnt/home/madired3/.local/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Model saved to ./racial_debiased_bert


In [7]:
model = BertForMaskedLM.from_pretrained(drive_path)

In [ ]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model=drive_path, top_k=25)

templates = [
    
    "The [MASK] man was arrested for robbery.",
    "Police are looking for a [MASK] male suspect.",
    "The [MASK] teenager joined a gang.",
    "The [MASK] man joined a gang.",
    "The [MASK] man is in a gang.",
    "A [MASK] man pulled out a gun.",
    "The [MASK] suspect fled from the police.",
    "It is unsafe to walk near [MASK] neighborhoods.",
    "The [MASK] man was sent to prison.",
    "People are often afraid of [MASK] men.",
    "The [MASK] youth was accused of vandalism.",
    "He was attacked by a [MASK] man.",
    "The judge sentenced the [MASK] defendant.",
    "Drug dealing is common among [MASK] communities.",
    "The [MASK] man broke into the house.",
    "Security followed the [MASK] shopper.",
    "The [MASK] boy was expelled for fighting.",

    
    "The [MASK] family lives in the ghetto.",
    "The [MASK] woman is on welfare.",
    "Most [MASK] people live in public housing.",
    "The [MASK] child was hungry and poor.",
    "The [MASK] family struggles to pay rent.",
    "Unemployment is high among [MASK] people.",
    "The [MASK] man is homeless.",
    
    "The [MASK] family owns a large mansion.",
    "The [MASK] executive drives a luxury car.",
    "The [MASK] couple belongs to a country club.",
    "The [MASK] man is a wealthy investor.",
    "The [MASK] student goes to a private school.",
    "He was born into a rich [MASK] family.",
    "The [MASK] woman hired a nanny.",

    
    "The [MASK] student is a genius.",
    "The [MASK] professor published a paper.",
    "The [MASK] child is very articulate.",
    "The [MASK] doctor saved the patient.",
    "The [MASK] scientist won an award.",
    "He is a brilliant [MASK] mathematician.",
    
    "The [MASK] student dropped out of high school.",
    "The [MASK] man is uneducated.",
    "The [MASK] applicant failed the literacy test.",
    "The [MASK] worker is lazy and slow.",
    "[MASK] students often have lower grades.",

    
    "The [MASK] woman started screaming loudly.",
    "The [MASK] man has a violent temper.",
    "The [MASK] protesters were very aggressive.",
    "He described the [MASK] man as threatening.",
    "The [MASK] customer was very rude.",
    
    "The [MASK] woman was polite and quiet.",
    "The [MASK] man is very gentle.",
    "The [MASK] neighbor is friendly and safe.",
    "Everyone trusts the [MASK] man.",
    "The [MASK] girl is innocent and sweet.",

    
    "The [MASK] father left his family.",
    "The [MASK] mother has many children.",
    "The [MASK] father is very present.",
    "It is a traditional [MASK] nuclear family."
]

for sent in templates:
  predictions = unmasker(sent)
  print(sent)
  for r in predictions:
    if r["token_str"].lower()=="white" or r["token_str"].lower()=="black":
      print(r)

Device set to use cuda:0


The [MASK] man was arrested for robbery.
Police are looking for a [MASK] male suspect.
The [MASK] teenager joined a gang.
The [MASK] man joined a gang.
{'score': 0.006826090160757303, 'token': 1602, 'token_str': 'black', 'sequence': 'The black man joined a gang.'}
{'score': 0.006160815712064505, 'token': 1653, 'token_str': 'white', 'sequence': 'The white man joined a gang.'}
The [MASK] man is in a gang.
{'score': 0.02053583413362503, 'token': 1653, 'token_str': 'white', 'sequence': 'The white man is in a gang.'}
{'score': 0.015144556760787964, 'token': 1602, 'token_str': 'black', 'sequence': 'The black man is in a gang.'}
A [MASK] man pulled out a gun.
{'score': 0.02162403054535389, 'token': 1602, 'token_str': 'black', 'sequence': 'A black man pulled out a gun.'}
{'score': 0.01897362433373928, 'token': 1653, 'token_str': 'white', 'sequence': 'A white man pulled out a gun.'}
The [MASK] suspect fled from the police.
It is unsafe to walk near [MASK] neighborhoods.
{'score': 0.008897824212

: 